In [1]:
import pyspark

spark = pyspark.sql.SparkSession.builder.appName("Graphs_Practice").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", spark._sc.defaultParallelism)
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 500)

import graphframes as gf

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 12:47:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### GraphFrames

Agenda:
* Creating vertices and edges
* Viewing properties of a GraphFrame
* Graph filtering
* Motifs - finding patterns
* Graph Algorithms

In [2]:
help(gf)
# GraphFrame = 2 dataframes: vertices and edges

Help on package graphframes:

NAME
    graphframes

PACKAGE CONTENTS
    examples (package)
    graphframe
    lib (package)
    tests

CLASSES
    builtins.object
        graphframes.graphframe.GraphFrame
    
    class GraphFrame(builtins.object)
     |  GraphFrame(v, e)
     |  
     |  Represents a graph with vertices and edges stored as DataFrames.
     |  
     |  :param v:  :class:`DataFrame` holding vertex information.
     |             Must contain a column named "id" that stores unique
     |             vertex IDs.
     |  :param e:  :class:`DataFrame` holding edge information.
     |             Must contain two columns "src" and "dst" storing source
     |             vertex IDs and destination vertex IDs of edges, respectively.
     |  
     |  >>> localVertices = [(1,"A"), (2,"B"), (3, "C")]
     |  >>> localEdges = [(1,2,"love"), (2,1,"hate"), (2,3,"follow")]
     |  >>> v = sqlContext.createDataFrame(localVertices, ["id", "name"])
     |  >>> e = sqlContext.createData

In [3]:
# Let's load in some sample data
fraud_df = spark.read.csv("data/paysim.csv", header=True, inferSchema=True)
display(fraud_df)

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
1,TRANSFER,181.0,C1305486145,181.0,0.0,C553264065,0.0,0.0,1,0
1,CASH_OUT,181.0,C840083671,181.0,0.0,C38997010,21182.0,0.0,1,0
1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0
1,PAYMENT,7817.71,C90045638,53860.0,46042.29,M573487274,0.0,0.0,0,0
1,PAYMENT,7107.77,C154988899,183195.0,176087.23,M408069119,0.0,0.0,0,0
1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.0,0,0
1,PAYMENT,4024.36,C1265012928,2671.0,0.0,M1176932104,0.0,0.0,0,0
1,DEBIT,5337.77,C712410124,41720.0,36382.23,C195600860,41898.0,40348.79,0,0


#### Vertices

* Needs to contain **id** column

In [4]:
import pyspark.sql.functions as F

fraud_vertices = (fraud_df
                  .select(F.col("nameOrig").alias("id"))
                  .union(fraud_df
                        .select(F.col("nameDest").alias("id")))
                  .distinct()
)

display(fraud_vertices)

id
C1305486145
C1912850431
C1648232591
C761750706
C504336483
C840514538
C768216420
C260084831
C484199463
C867288517


#### Edges

* Needs to contain **src** and **dst** columns

In [7]:
fraud_edges = (fraud_df
               .withColumnRenamed("nameOrig","src")
               .withColumnRenamed("nameDest","dst")
)

display(fraud_edges)

step,type,amount,src,oldbalanceOrg,newbalanceOrig,dst,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
1,TRANSFER,181.0,C1305486145,181.0,0.0,C553264065,0.0,0.0,1,0
1,CASH_OUT,181.0,C840083671,181.0,0.0,C38997010,21182.0,0.0,1,0
1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0
1,PAYMENT,7817.71,C90045638,53860.0,46042.29,M573487274,0.0,0.0,0,0
1,PAYMENT,7107.77,C154988899,183195.0,176087.23,M408069119,0.0,0.0,0,0
1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.0,0,0
1,PAYMENT,4024.36,C1265012928,2671.0,0.0,M1176932104,0.0,0.0,0,0
1,DEBIT,5337.77,C712410124,41720.0,36382.23,C195600860,41898.0,40348.79,0,0


In [8]:
# Let's create our first GraphFrame

fraud_graph = gf.GraphFrame(fraud_vertices, fraud_edges)

fraud_vertices.cache()
fraud_edges.cache()

display(fraud_graph)

26/04/26 12:47:58 WARN CacheManager: Asked to cache already cached data.
26/04/26 12:47:58 WARN CacheManager: Asked to cache already cached data.


GraphFrame(v:[id: string], e:[src: string, dst: string ... 9 more fields])

#### Viewing properties of a GraphFrame

In [9]:
# All of these return a Spark DataFrame

display(fraud_graph.vertices) # same as our created dataframe
# display(fraud_graph.edges) # same as our created edges
# display(fraud_graph.degrees) # total edges connected to a vertice
# display(fraud_graph.inDegrees) # incoming edges
# display(fraud_graph.outDegrees) # outgoing edges
# display(fraud_graph.triplets) # source / edge / destination combined

26/04/26 12:48:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/26 12:48:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/26 12:48:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/26 12:48:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/26 12:48:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/26 12:48:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/26 12:48:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/26 12:48:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/26 12:48:00 WARN RowBasedKeyValueBatch: Calling spill() on

id
C1305486145
C1912850431
C1648232591
C761750706
C504336483
C840514538
C768216420
C260084831
C484199463
C867288517


### Graph filtering

In [10]:
# filtering vertices
fraud_graph_filtered_v = fraud_graph.filterVertices("id == 'C1420196421'")
display(fraud_graph_filtered_v.vertices) # only this vertex
#display(fraud_graph_filtered_v.edges) # no edges, because just having this vertex does not "contain" any edge

# fraud_graph_filtered_v = fraud_graph.filterVertices("id == 'C1420196421' OR id == 'C972765878'") # now we have two connected vertices
# display(fraud_graph_filtered_v.vertices)
# display(fraud_graph_filtered_v.edges) 

/usr/local/lib/python3.10/dist-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


id
C1420196421


In [11]:
# filtering edges
fraud_graph_filtered_e = fraud_graph.filterEdges("isFraud == 1")
display(fraud_graph_filtered_e.vertices)
# display(fraud_graph_filtered_e.edges)

id
C1305486145
C1912850431
C1648232591
C761750706
C504336483
C840514538
C768216420
C260084831
C484199463
C867288517


In [12]:
# note that we still have all the vertices, even if they are not on an edge with "isFraud"
print(f"""
Count of edges: {fraud_graph_filtered_e.edges.count()}
Count of vertices: {fraud_graph_filtered_e.vertices.count()}
""")
      
print(f"Original fraud edge count: {fraud_edges.filter('isFraud == 1').count()}")

# If you want to remove orphaned vertices, combine it with dropIsolatedVertices()

# fraud_graph_filtered_e_clean = (fraud_graph
#                                 .filterEdges("isFraud == 1")
#                                 .dropIsolatedVertices()
#                                )
# print(f"""
# Count of edges: {fraud_graph_filtered_e_clean.edges.count()}
# Count of vertices: {fraud_graph_filtered_e_clean.vertices.count()}
# """)
  


Count of edges: 2343
Count of vertices: 3817022

Original fraud edge count: 2343


### Motifs

(vertice)-[edge]->(vertice)

In [13]:
# The naming of vertice/edge - used for mapping to a specific identity (note that b is definitely b in the example below, but c may or may not be same as a)
# Semicolon for bundling multiple patterns
# If we want to apply some filters, we should apply them on the resulting dataframe (eg isFraud below)

money_launderers_df = (fraud_graph
                       .find("(a)-[e1]->(b); (b)-[e2]->(c)")
                       .filter(("e1.isFraud == 1 & e2.isFraud == 0"))
                      )

display(money_launderers_df)

a,e1,b,e2,c
{C1829721095},"{44, CASH_OUT, 534255.94, C1829721095, 534255.94, 0.0, C991247178, 24559.0, 558814.94, 1, 0}",{C991247178},"{162, CASH_OUT, 60387.69, C991247178, 0.0, 0.0, C1344561835, 2076100.86, 2347101.49, 0, 0}",{C1344561835}
{C1003023037},"{74, CASH_OUT, 158489.29, C1003023037, 158489.29, 0.0, C1969565765, 0.0, 158489.29, 1, 0}",{C1969565765},"{206, PAYMENT, 3718.53, C1969565765, 82266.0, 78547.47, M657628830, 0.0, 0.0, 0, 0}",{M657628830}
{C145966586},"{66, CASH_OUT, 548269.98, C145966586, 548269.98, 0.0, C979594589, 0.0, 548269.98, 1, 0}",{C979594589},"{204, CASH_OUT, 33336.76, C979594589, 0.0, 0.0, C960842494, 76321.69, 109658.45, 0, 0}",{C960842494}


In [14]:
# We can have empty brackets - then this entity is left out of the resulting dataframe

outgoing_edges_df = (fraud_graph
                       .find("(a)-[edge]->()")
                      )

display(outgoing_edges_df)

a,edge
{C1000003372},"{167, CASH_IN, 20528.65, C1000003372, 2302074.12, 2322602.77, C1840417793, 82696.17, 62167.52, 0, 0}"
{C1000004530},"{41, CASH_OUT, 93865.13, C1000004530, 351422.72, 257557.59, C1643839147, 178083.14, 271948.26, 0, 0}"
{C1000037689},"{10, CASH_IN, 451174.06, C1000037689, 4042922.49, 4494096.56, C925108082, 1900737.34, 1449563.28, 0, 0}"
{C1000041967},"{155, PAYMENT, 16675.57, C1000041967, 146490.0, 129814.43, M567997887, 0.0, 0.0, 0, 0}"
{C1000042392},"{38, TRANSFER, 666487.42, C1000042392, 0.0, 0.0, C418760645, 1627959.34, 2294446.77, 0, 0}"
{C1000062907},"{132, CASH_OUT, 130635.27, C1000062907, 81756.0, 0.0, C1982753296, 0.0, 132980.75, 0, 0}"
{C1000063018},"{153, TRANSFER, 150384.96, C1000063018, 192.0, 0.0, C1483225154, 109918.83, 260303.79, 0, 0}"
{C100006673},"{11, CASH_OUT, 363683.6, C100006673, 0.0, 0.0, C1445811732, 963034.71, 1283588.67, 0, 0}"
{C1000073191},"{20, CASH_OUT, 185987.72, C1000073191, 0.0, 0.0, C216084411, 1575292.64, 1573563.04, 0, 0}"
{C1000075512},"{187, CASH_OUT, 29971.59, C1000075512, 101014.0, 71042.41, C205621775, 0.0, 29971.59, 0, 0}"


## Graph Algorithms

### PageRank

In [ ]:
# graphframes.examples is broken on PySpark 3.x (imports removed pyspark.tests).
# Define the friends graph inline — same data as the original example.
v = spark.createDataFrame([
    ("a", "Alice",   34),
    ("b", "Bob",     36),
    ("c", "Charlie", 30),
    ("d", "David",   29),
    ("e", "Esther",  32),
    ("f", "Fanny",   36),
    ("g", "Gabby",   60),
], ["id", "name", "age"])

e = spark.createDataFrame([
    ("a", "b", "friend"),
    ("b", "c", "follow"),
    ("c", "b", "follow"),
    ("f", "c", "follow"),
    ("e", "f", "follow"),
    ("e", "d", "friend"),
    ("d", "a", "friend"),
    ("a", "e", "friend"),
], ["src", "dst", "relationship"])

g = gf.GraphFrame(v, e)
display(g.vertices)
display(g.edges)
display(g.triplets)

In [ ]:
g_pagerank = g.pageRank(resetProbability=0.15, maxIter=10)
display(g_pagerank.vertices)
display(g_pagerank.edges)

### Triangle count (3-clique)

In [ ]:
g_trianglecount = g.triangleCount()
display(g_trianglecount)

In [ ]:
# Let's add a few more edges to see some triangles

new_edges = [
  {"src": "c",
  "dst": "a"
  },
  {"src": "c",
   "dst": "e" 
  }
]

new_edges_df = spark.createDataFrame(new_edges)

all_edges_df = (g.edges
               .unionByName(new_edges_df, allowMissingColumns=True))

new_g = gf.GraphFrame(g.vertices, all_edges_df)

display(new_g.triplets)

In [ ]:
new_g_trianglecount = new_g.triangleCount()
display(new_g_trianglecount)

### Label propagation

In [ ]:
g_labelprop = g.labelPropagation(maxIter=5)
display(g_labelprop)

In [ ]:
# This algorithm is computationally efficient, but not always very useful. E.g.:
new_g_labelprop = new_g.labelPropagation(maxIter=5)
display(new_g_labelprop)

### Breadth-first search

In [ ]:
g_bfs = g.bfs("name = 'Esther'", "age < 32")
display(g_bfs)

In [ ]:
new_g_bfs = new_g.bfs("name = 'Esther'", "age > 29 and age < 36 and name != 'Esther'")
display(new_g_bfs)

### Further reading

https://graphframes.github.io/graphframes/docs/_site/user-guide.html  
https://docs.databricks.com/spark/latest/graph-analysis/graphframes/user-guide-python.html  
https://blog.devgenius.io/graph-modeling-in-pyspark-using-graphframes-part-1-e7cb42099182

### Task
Dataset: Star Wars Social Network  
source: https://www.kaggle.com/datasets/ruchi798/star-wars?resource=download&select=starwars-full-interactions-allCharacters-merged.json

Todo:
* Preprocess the data:
  * create a dataframe for vertices
  * create a dataframe for edges. Please make the edges undirected 
    * hint: the data is currently directed, but actually the direction has no meaning for this dataset
* Create a GraphFrame using the vertices and undirected edges
* Run pagerank on top of the GraphFrame. 
  * Order by pagerank, descending. Discuss, why do the values and pageranks not correlate across characters?
* Using motifs, find characters who never appear together with Luke but appear at least 5 times together with a character that appears at least once with Luke. 
  * E.g. Luke never appears together in a scene with Padme, but both characters appear on scenes with R2-D2.

In [ ]:
# Your solution:
# dataset: input/starwars_full_interactions_allCharacters_merged.json.json